In [ ]:
import pandas as pd
import numpy as np
import re
from collections import OrderedDict

# ----------------------------------------------------------------------
# 1. Load the rules CSV
# ----------------------------------------------------------------------
rules_path = r"E:\Abroad period research\New idea for 2026\Corn Yield Estimation\Final codes\paper_visualizations_fixed\model_logic_rules.csv"
df_rules = pd.read_csv(rules_path)

print(f"Loaded {len(df_rules)} rules from {rules_path}")
print("Columns:", df_rules.columns.tolist())

# Ensure we have the string representations of antecedents and consequents
if 'antecedents_str' not in df_rules.columns:
    # Reconstruct from the list columns if needed
    df_rules['antecedents_str'] = df_rules['antecedents'].apply(
        lambda x: ' | '.join([i.replace('F_', '') for i in eval(x)]) if isinstance(x, str) else ''
    )
if 'consequents_str' not in df_rules.columns:
    df_rules['consequents_str'] = df_rules['consequents'].apply(
        lambda x: ' | '.join([i.replace('S_', '') for i in eval(x)]) if isinstance(x, str) else ''
    )

# If the rule_strength column is missing, compute it (should be present)
if 'rule_strength' not in df_rules.columns:
    df_rules['rule_strength'] = df_rules['lift'] * df_rules['confidence']
    # Apply boost for lift > 1.5
    df_rules.loc[df_rules['lift'] > 1.5, 'rule_strength'] *= 1.2

# ----------------------------------------------------------------------
# 2. Define domain sets (based on feature name patterns)
# ----------------------------------------------------------------------
# Satellite: NDVI, EVI, GPP, and their derivatives (seasonal, etc.)
SAT_KEYWORDS = ['NDVI', 'EVI', 'GPP', 'NIRv']
# Climate: temperature, precipitation, VPD, dew point, etc.
CLIM_KEYWORDS = ['TMIN', 'TMAX', 'TMEAN', 'TDMEAN', 'PPT', 'VPD', 'VPDMIN', 'VPDMAX']
# Soil: organic matter, clay, sand, pH, AWC, AWS, CEC, etc.
SOIL_KEYWORDS = ['organic_matter', 'clay', 'sand', 'pH', 'awc', 'aws', 'cec', 'field_capacity', 
                 'wilting_point', 'saturated_hc', 'b_density']
# Geographic: X, Y, year, possibly state
GEO_KEYWORDS = ['X', 'Y', 'year', 'STATE']

def get_domain(feature_name):
    """
    Map a clean feature name (e.g., 'NDVI_8', 'organic_matter') to a domain.
    Returns one of: 'Sat', 'Clim', 'Soil', 'Geo', or 'Other'.
    """
    f_upper = feature_name.upper()
    # Check Sat first
    for kw in SAT_KEYWORDS:
        if kw in f_upper:
            return 'Sat'
    # Then Clim
    for kw in CLIM_KEYWORDS:
        if kw in f_upper:
            return 'Clim'
    # Then Soil
    for kw in SOIL_KEYWORDS:
        if kw in f_upper:
            return 'Soil'
    # Then Geo
    for kw in GEO_KEYWORDS:
        if kw in f_upper:
            return 'Geo'
    return 'Other'

# ----------------------------------------------------------------------
# 3. Parse antecedents and consequents into lists of feature names
# ----------------------------------------------------------------------
def parse_items(concat_str):
    """Split concatenated string (e.g., 'NDVI_8_Low | GPP_8_Low') into list of clean feature names."""
    if pd.isna(concat_str) or concat_str == '':
        return []
    # Remove any leading/trailing spaces and split on '|'
    parts = [p.strip() for p in concat_str.split('|') if p.strip()]
    # Each part is like 'NDVI_8_Low' – we need the base feature name without the bin suffix.
    # The bin suffix is always the last underscore part: _Low, _Medium, _High, or _PosImpact, etc.
    # We'll strip the last part after the last underscore.
    clean_features = []
    for p in parts:
        # Remove the bin label (everything after the last underscore)
        if '_' in p:
            # But some names have underscores in the base (e.g., 'organic_matter_Low')
            # We split on last underscore only.
            base = '_'.join(p.split('_')[:-1])
            clean_features.append(base)
        else:
            clean_features.append(p)
    return clean_features

df_rules['antecedent_features'] = df_rules['antecedents_str'].apply(parse_items)
df_rules['consequent_features'] = df_rules['consequents_str'].apply(parse_items)

# ----------------------------------------------------------------------
# 4. Classify each rule according to Algorithm 3
# ----------------------------------------------------------------------
def classify_rule(ante_features, cons_features):
    # Get domains for each feature (unique)
    ante_domains = set()
    for f in ante_features:
        d = get_domain(f)
        if d != 'Other':
            ante_domains.add(d)
    cons_domains = set()
    for f in cons_features:
        d = get_domain(f)
        if d != 'Other':
            cons_domains.add(d)
    
    # According to Algorithm 3:
    # - Discovery: |Adom| == 1 and |Bdom| == 1 and Adom ∩ Bdom == ∅
    if len(ante_domains) == 1 and len(cons_domains) == 1 and not (ante_domains & cons_domains):
        return 'Discovery'
    # - Consistency: Adom ∩ Bdom != ∅ (overlap)
    elif (ante_domains & cons_domains):
        return 'Consistency'
    # - Mixed: all other cases
    else:
        return 'Mixed'

df_rules['category'] = df_rules.apply(
    lambda row: classify_rule(row['antecedent_features'], row['consequent_features']),
    axis=1
)

# Also compute a domain mapping string (like "Sat → Clim") for display
def domain_mapping_str(ante_features, cons_features):
    ante_domains = {get_domain(f) for f in ante_features if get_domain(f) != 'Other'}
    cons_domains = {get_domain(f) for f in cons_features if get_domain(f) != 'Other'}
    if not ante_domains or not cons_domains:
        return 'Other'
    return ' / '.join(sorted(ante_domains)) + ' → ' + ' / '.join(sorted(cons_domains))

df_rules['domain_mapping'] = df_rules.apply(
    lambda row: domain_mapping_str(row['antecedent_features'], row['consequent_features']),
    axis=1
)

# ----------------------------------------------------------------------
# 5. Sort each category by rule_strength and select top 20 (or all)
# ----------------------------------------------------------------------
top_n = 20  # as shown in the paper tables
categories = ['Discovery', 'Consistency', 'Mixed']

for cat in categories:
    df_cat = df_rules[df_rules['category'] == cat].sort_values('rule_strength', ascending=False)
    print(f"\n{cat} rules: {len(df_cat)}")
    if len(df_cat) > 0:
        # Export top N
        top_df = df_cat.head(top_n)
        # Prepare columns for export: include relevant info
        export_cols = ['antecedents_str', 'consequents_str', 'support', 'confidence', 
                       'lift', 'rule_strength', 'domain_mapping', 'features']
        # If the original had 'features', we keep it; else compute from both sides
        if 'features' not in top_df.columns:
            top_df['features'] = top_df.apply(
                lambda row: list(set(row['antecedent_features'] + row['consequent_features'])),
                axis=1
            )
        # Save CSV
        out_path = f"paper_visualizations_fixed/{cat.lower()}_rules_top{top_n}.csv"
        top_df[export_cols].to_csv(out_path, index=False)
        print(f"  Saved top {len(top_df)} to {out_path}")
    else:
        print(f"  No rules in this category.")

# Also save all rules with category and domain mapping for reference
df_rules[['antecedents_str', 'consequents_str', 'support', 'confidence', 'lift', 
          'rule_strength', 'category', 'domain_mapping']].to_csv(
    'paper_visualizations_fixed/rules_with_categories.csv', index=False
)
print("\n✅ All rules with categories saved to rules_with_categories.csv")

# ----------------------------------------------------------------------
# 6. Print summary
# ----------------------------------------------------------------------
print("\n" + "="*60)
print("CATEGORIZATION SUMMARY")
print("="*60)
for cat in categories:
    count = len(df_rules[df_rules['category'] == cat])
    print(f"{cat:12s}: {count:5d} rules")
print("="*60)

# Show a few examples from each category
for cat in categories:
    print(f"\n--- Top 5 {cat} Rules ---")
    sample = df_rules[df_rules['category'] == cat].sort_values('rule_strength', ascending=False).head(5)
    for idx, row in sample.iterrows():
        print(f"  Ant: {row['antecedents_str'][:50]}...")
        print(f"  Con: {row['consequents_str'][:50]}...")
        print(f"  Strength: {row['rule_strength']:.3f}, Domain: {row['domain_mapping']}")
        print()